# CS-245 Machine Learning Project
## Phase 3: Unsupervised Learning — K-Means Clustering

---

**Group Members:** 

Sukaina Nasir

Zayna Qasim

Hamna Shah


**Course:** CS-245 — Machine Learning

---

In Phase 2 we built supervised models that predict property prices. In this phase we shift to unsupervised learning, where we don't have a label to predict — instead we're trying to find hidden structure in the data on our own.

We're using **K-Means Clustering** to group properties into natural segments based on their features. The idea is to see whether the data naturally separates into meaningful groups like "budget properties", "mid-range", and "luxury" without us telling the model anything about prices upfront.

This is required by the project guidelines: at least one unsupervised learning approach with silhouette score and visualisation.

Make sure `processed_data.csv` from Phase 1 is uploaded before running this.


---
## Step 0 — Imports

We need a few extra things compared to Phase 2 — mainly the clustering and dimensionality reduction tools.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Clustering
from sklearn.cluster import KMeans

# Evaluation
from sklearn.metrics import silhouette_score, silhouette_samples

# Dimensionality reduction for visualisation
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f8f9fa",
    "axes.grid":        True,
    "grid.color":       "#e0e0e0",
    "grid.linestyle":   "--",
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})

PALETTE = ["#2E86AB", "#F18F01", "#A23B72", "#44BBA4", "#E94F37", "#6B4226"]

print("Libraries loaded.")


---
## Step 1 — Load the Data

Same processed dataset from Phase 1.


In [ ]:
df = pd.read_csv("processed_data.csv")

print(f"Shape: {df.shape}")
df.head()


---
## Step 2 — Select Features for Clustering

Not all features are equally useful for clustering. We want features that describe the property's physical and financial profile. We're using:

- `area_marla` — size of the property
- `bedroom` — number of bedrooms
- `bath` — number of bathrooms
- `price_lakh` — price (we include this since clustering is unsupervised — there's no "label" we're cheating off of)
- `city_enc` — location
- `type_enc` — property type

The goal is to see if K-Means can segment properties into groups that make real-world sense — like budget apartments vs mid-range houses vs expensive villas — without us manually defining those categories.


In [ ]:
CLUSTER_FEATURES = ["area_marla", "bedroom", "bath", "price_lakh", "city_enc", "type_enc"]

X_cluster = df[CLUSTER_FEATURES].copy()

print(f"Clustering on {len(CLUSTER_FEATURES)} features, {len(X_cluster)} samples.")
print()
print(X_cluster.describe().round(2))


---
## Step 3 — Scale the Features

K-Means uses Euclidean distance to assign points to clusters. This means features with larger ranges will dominate the distance calculation.

For example, `price_lakh` can be in the hundreds while `bedroom` is between 1 and 10. Without scaling, the algorithm would basically just cluster by price and ignore everything else.

StandardScaler fixes this by bringing all features to the same scale.


In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

X_scaled_df = pd.DataFrame(X_scaled, columns=CLUSTER_FEATURES)

print("Feature stats after scaling (all should have mean ~0, std ~1):")
print(X_scaled_df.describe().round(3))


---
## Step 4 — Finding the Right Number of Clusters: Elbow Method

K-Means requires us to specify the number of clusters (K) beforehand. We don't always know what the right number is, so we use the **Elbow Method**.

We run K-Means for K = 2 to 10, and for each K we record the **inertia** — the sum of squared distances from each point to its assigned cluster center. As K increases, inertia always decreases. We look for the "elbow" — the point where adding more clusters stops giving much benefit.

It's not always an obvious elbow, but it gives us a good range to investigate.


In [ ]:
inertia_values = []
k_range = range(2, 11)

print("Running K-Means for K = 2 to 10...")
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia_values.append(km.inertia_)
    print(f"  K={k}  ->  Inertia: {km.inertia_:.0f}")

print("Done.")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(k_range), inertia_values, marker="o", color=PALETTE[0],
        linewidth=2, markersize=7, markerfacecolor="white", markeredgewidth=2)
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Inertia (Within-Cluster Sum of Squares)")
ax.set_title("Elbow Method — Finding Optimal K")
ax.set_xticks(list(k_range))

plt.tight_layout()
plt.savefig("plot13_elbow_method.png", dpi=150, bbox_inches="tight")
plt.show()

print("Look for the point where the curve bends like an elbow.")
print("Beyond that point, increasing K gives diminishing returns.")


---
## Step 5 — Silhouette Score Analysis

The Elbow Method is somewhat subjective. The **Silhouette Score** gives a more quantitative answer.

For each data point, the silhouette score measures:
- How similar it is to its own cluster (cohesion)
- How different it is from the nearest other cluster (separation)

The score ranges from -1 to 1:
- Close to 1 means the point is well-matched to its cluster
- Close to 0 means it's on the boundary between clusters
- Negative means it might be assigned to the wrong cluster

We want the K with the highest average silhouette score.


In [ ]:
silhouette_scores = []

print("Computing silhouette scores for K = 2 to 10...")
for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"  K={k}  ->  Silhouette Score: {score:.4f}")

best_k = list(k_range)[silhouette_scores.index(max(silhouette_scores))]
print(f"\nBest K by silhouette score: K = {best_k}  (score = {max(silhouette_scores):.4f})")


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(list(k_range), silhouette_scores, marker="s", color=PALETTE[2],
        linewidth=2, markersize=7, markerfacecolor="white", markeredgewidth=2)
ax.axvline(x=best_k, color="red", linestyle="--", alpha=0.7, label=f"Best K = {best_k}")
ax.set_xlabel("Number of Clusters (K)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Silhouette Score by Number of Clusters")
ax.set_xticks(list(k_range))
ax.legend()

plt.tight_layout()
plt.savefig("plot14_silhouette_scores.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Step 6 — Train the Final K-Means Model

Based on the Elbow and Silhouette analysis, we choose our K and train the final clustering model.

We're using `n_init=10` which means K-Means runs 10 times with different random starting points and keeps the best result. This reduces the chance of getting stuck in a bad local minimum.


In [ ]:
# Use the best K found above — or override manually if needed
K = best_k

print(f"Training final K-Means model with K = {K}...")

kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)

print(f"Clustering done.")
print(f"\nCluster distribution:")
print(df["cluster"].value_counts().sort_index())


In [ ]:
# Final silhouette score on the chosen model
final_score = silhouette_score(X_scaled, df["cluster"])
print(f"Final Silhouette Score (K={K}): {final_score:.4f}")
print()
print("Interpretation:")
print("  0.71 - 1.00  ->  Strong structure")
print("  0.51 - 0.70  ->  Reasonable structure")
print("  0.26 - 0.50  ->  Weak structure (clusters may overlap)")
print("  < 0.26       ->  No substantial structure found")


---
## Step 7 — Understanding the Clusters

The whole point of clustering here is to see if the groups make real-world sense. We look at the mean values of each feature per cluster to give each cluster a label.


In [ ]:
cluster_profile = df.groupby("cluster")[CLUSTER_FEATURES].mean().round(2)
print("Cluster Profiles (mean values per cluster):")
print(cluster_profile.to_string())


In [ ]:
# Sort clusters by average price to assign meaningful labels
price_order = cluster_profile["price_lakh"].sort_values()
print("Clusters sorted by mean price (Lakhs):")
print(price_order)
print()
print("From this we can label clusters roughly as:")
for rank, (idx, price) in enumerate(price_order.items()):
    if rank == 0:
        label = "Budget / Low-Range"
    elif rank == len(price_order) - 1:
        label = "Luxury / High-End"
    else:
        label = f"Mid-Range Segment {rank}"
    print(f"  Cluster {idx}  ->  Mean Price: {price:.1f} Lakhs  ->  {label}")


---
## Step 8 — Visualisations

### Plot 1: PCA Scatter — Clusters in 2D

K-Means clusters in 6-dimensional space. We can't visualise 6 dimensions directly, so we use **PCA (Principal Component Analysis)** to compress the data down to 2 dimensions while keeping as much variance as possible.

This is only for visualisation — the actual clustering was done in the full 6D space.


In [ ]:
# Reduce to 2D using PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"Variance explained by PC1: {explained[0]*100:.1f}%")
print(f"Variance explained by PC2: {explained[1]*100:.1f}%")
print(f"Total variance captured  : {sum(explained)*100:.1f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for cluster_id in sorted(df["cluster"].unique()):
    mask = df["cluster"] == cluster_id
    ax.scatter(
        X_pca[mask, 0], X_pca[mask, 1],
        label=f"Cluster {cluster_id}",
        alpha=0.4, s=12,
        color=PALETTE[cluster_id % len(PALETTE)],
        edgecolors="none"
    )

# Plot cluster centers in PCA space
centers_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centers_pca[:, 0], centers_pca[:, 1],
           c="black", marker="X", s=150, zorder=5, label="Cluster Centers")

ax.set_xlabel(f"Principal Component 1 ({explained[0]*100:.1f}% variance)")
ax.set_ylabel(f"Principal Component 2 ({explained[1]*100:.1f}% variance)")
ax.set_title(f"K-Means Clusters Visualised in 2D (via PCA)  |  K = {K}")
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("plot15_pca_clusters.png", dpi=150, bbox_inches="tight")
plt.show()

print("Note: PCA compresses 6D data to 2D for visualisation only.")
print("Some cluster overlap in 2D is expected — they may be well-separated in higher dimensions.")


### Plot 2: Silhouette Plot

A silhouette plot shows the silhouette score for every individual data point, grouped by cluster. Wider bars = more data points. We want bars extending to the right — that means points are well-matched to their clusters.


In [ ]:
silhouette_vals = silhouette_samples(X_scaled, df["cluster"])

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for i in range(K):
    cluster_sil_vals = silhouette_vals[df["cluster"] == i]
    cluster_sil_vals.sort()
    size = len(cluster_sil_vals)
    y_upper = y_lower + size

    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0, cluster_sil_vals,
        alpha=0.7,
        color=PALETTE[i % len(PALETTE)],
        label=f"Cluster {i}"
    )
    ax.text(-0.05, y_lower + 0.5 * size, str(i), fontsize=9)
    y_lower = y_upper + 10

ax.axvline(x=final_score, color="red", linestyle="--", linewidth=1.5,
           label=f"Avg Silhouette = {final_score:.3f}")
ax.set_xlabel("Silhouette Coefficient")
ax.set_ylabel("Cluster")
ax.set_title(f"Silhouette Plot — K = {K}")
ax.set_yticks([])
ax.legend(loc="upper right")

plt.tight_layout()
plt.savefig("plot16_silhouette_plot.png", dpi=150, bbox_inches="tight")
plt.show()


### Plot 3: Cluster Profiles — Feature Comparison

This heatmap shows the mean value of each feature per cluster, after scaling. It makes it easy to see what makes each cluster different.


In [ ]:
# Compute scaled cluster profiles
cluster_scaled_profiles = pd.DataFrame(
    scaler.transform(cluster_profile),
    columns=CLUSTER_FEATURES,
    index=[f"Cluster {i}" for i in cluster_profile.index]
)

fig, ax = plt.subplots(figsize=(10, max(4, K)))
sns.heatmap(
    cluster_scaled_profiles,
    annot=True, fmt=".2f",
    cmap="RdYlGn", center=0,
    linewidths=0.5, linecolor="white",
    ax=ax, annot_kws={"size": 10}
)
ax.set_title("Cluster Profiles — Scaled Feature Means")
ax.set_xlabel("Feature")
ax.set_ylabel("Cluster")

plt.tight_layout()
plt.savefig("plot17_cluster_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("Red = below average for that feature. Green = above average.")
print("This tells us the personality of each cluster.")


### Plot 4: Price Distribution per Cluster

This is probably the most intuitive plot — it shows how prices are distributed within each cluster. If clustering worked well, these distributions should be clearly separated.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for cluster_id in sorted(df["cluster"].unique()):
    subset = df[df["cluster"] == cluster_id]["price_lakh"]
    ax.hist(subset, bins=50, alpha=0.6,
            color=PALETTE[cluster_id % len(PALETTE)],
            label=f"Cluster {cluster_id}  (n={len(subset):,})",
            edgecolor="white")

ax.set_xlabel("Price (Lakhs)")
ax.set_ylabel("Number of Properties")
ax.set_title("Price Distribution by Cluster")
ax.legend()

plt.tight_layout()
plt.savefig("plot18_cluster_price_distribution.png", dpi=150, bbox_inches="tight")
plt.show()


### Plot 5: Area vs Price, Coloured by Cluster

Scatter plot of area vs price with each cluster shown in a different colour. If the clusters represent meaningful property segments, we should see clear groupings here.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

sample_idx = df.sample(min(6000, len(df)), random_state=42).index

for cluster_id in sorted(df["cluster"].unique()):
    mask = (df["cluster"] == cluster_id) & df.index.isin(sample_idx)
    ax.scatter(
        df.loc[mask, "area_marla"],
        df.loc[mask, "price_lakh"],
        label=f"Cluster {cluster_id}",
        alpha=0.4, s=12,
        color=PALETTE[cluster_id % len(PALETTE)],
        edgecolors="none"
    )

ax.set_xlabel("Area (Marla)")
ax.set_ylabel("Price (Lakhs)")
ax.set_title("Area vs Price — Coloured by Cluster")
ax.legend()

plt.tight_layout()
plt.savefig("plot19_area_price_clusters.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Step 9 — Summary

### What we did

We applied K-Means Clustering to the property dataset to find natural groupings without using any labels. The steps were:

1. Selected relevant features (area, bedrooms, bathrooms, price, city, type)
2. Scaled features using StandardScaler so distances are meaningful
3. Used the Elbow Method and Silhouette Scores to find the best K
4. Trained the final model and evaluated it
5. Visualised clusters using PCA, silhouette plots, and feature heatmaps

### Results

The final silhouette score and cluster profiles are printed above after running the cells.

From the cluster profiles, the groups roughly correspond to:
- Lower-priced, smaller properties (budget segment)
- Mid-range properties with average area and bedrooms
- Larger, higher-priced properties (luxury segment)

This makes intuitive sense for Pakistani real estate where price segmentation is quite distinct across cities and property types.

### Connection to Phase 2

In Phase 2, we predicted the exact price using supervised models. Here we took a different approach and asked the question: without looking at price as a target, do properties naturally group themselves? The answer is yes — and the groups align well with the price predictions from Phase 2, which gives us more confidence that our features are meaningful.

---
Next: Phase 4 — Final Comparative Analysis and Report writing.
